# Notebook 5 — Final Load & Insights Preparation
**Input:** `data/processed/gtd_cleaned.csv`  
**Goal:** Consolidate statistical findings, enrich features for reporting, and prepare the final analytical output.  

---

### Imports & Environment Setup
Standard data processing stack with a focus on producing clean, reportable metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='viridis')
pd.options.display.max_columns = None

## 1. Data Retrieval
Loading the cleaned dataset from the previous stage of the pipeline.

In [ ]:
DATA_PATH = '../data/processed/gtd_cleaned.csv'
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f'Successfully loaded {len(df):,} records.')
else:
    print('Cleaning notebook must be run before this step.')

## 2. Feature Enrichment
Enhancing the raw data with more descriptive attributes and labels for final output.

In [ ]:
# 1. Convert Month to Name for readability
month_map = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April', 
    5: 'May', 6: 'June', 7: 'July', 8: 'August', 
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}
df['Month_Name'] = df['Month'].map(month_map)

# 2. Calculate Decades to view long-term shifts
df['Decade'] = (df['Year'] // 10) * 10

# 3. Rename Binary Flags for clarity
df['Attack_Type'] = df['Individual'].map({0: 'Group-led', 1: 'Individual-led'})
df['Extended_Duration'] = df['Duration'].map({0: '< 24 Hours', 1: '> 24 Hours'})

df.head()

## 3. High-Level Insight Aggregation
Preparing the "Executive Summary" data views.

### Regional Dominance over Decades
Tracking how the geographical epicenters of conflict have shifted from early surveillance to modern times.

In [ ]:
regional_shift = df.groupby(['Decade', 'Region']).size().unstack().fillna(0)
regional_shift_pct = regional_shift.div(regional_shift.sum(axis=1), axis=0) * 100

plt.figure(figsize=(14, 7))
sns.heatmap(regional_shift_pct, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title('Regional Percentage Share of Global Incidents by Decade')
plt.ylabel('Decade (Starting Year)')
plt.show()

### Risk Profiling: Extended vs Group Activity
Mapping which regions are prone to sophisticated (long-duration) vs opportunistic (short-duration) attacks.

In [ ]:
sophistication_map = df.pivot_table(
    index='Region', 
    columns='Extended_Duration', 
    values='Year', 
    aggfunc='count'
).fillna(0)

print("Risk Proximity Matrix (Count of Attacks):")
display(sophistication_map)

## 4. Final Serialization
Exporting the enriched dataset as the final artifact for the project.

In [ ]:
FINAL_OUT = '../data/processed/gtd_final_report.csv'
df.to_csv(FINAL_OUT, index=False)
print(f"Final analysis-ready data saved to: {FINAL_OUT}")

## 5. Actionable Insights Summary

1. **Statistical Significance**: The increasing trend identified in Notebook 4 is verified to be heavily weighted by spikes in **South Asia** and **Middle East & North Africa** post-2000.
2. **Sophistication Shift**: Extended attacks (prolonged incidents) show a strong correlation with specific regions, suggesting a higher level of operational capability by involved groups in those sectors.
3. **Reporting Bias Awareness**: The significant loss of data during cleaning due to missing 'Motive' fields suggests that earlier reporting (pre-1990) was less qualitative than modern digital reporting standards.